# DACFE_Ex: Tsai-Wu failure criterion
#### **(by Norbert Blanco)**

This notebook computes and plots the value of the Tsai-Wu criterion in a composite plate with UD plies subjected to both in-plane and out-of-plane loading.

Converted from the original MATLAB Live Script (`DACFE_Ex401.m`) to Python. It runs in Jupyter and in Google Colab without any additional setup (only `numpy`, `pandas` and `matplotlib`, all preinstalled in Colab).

## Initialization

Import the packages used throughout the notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=False)


## Definition of material, LSS and applied loads

#### Definition of material properties

- The material is assumed to be transversely isotropic and only 4 elastic constants are required: E11 (MPa), E22 (MPa), nu12, G12 (MPa).
- The plies are assumed to be working under plane-stress condition and 5 strength properties are required (all in absolute value): X11t (MPa), X11c (MPa), X22t (MPa), X22c (MPa) and X12 (MPa).
- CFRP T300/5208 is defined as material 1. Additional materials can be defined following the same structure `Mat[mat_num] = [E11, E22, nu12, G12, X11t, X11c, X22t, X22c, X12]`, or by updating the material properties.

In [ ]:
# Mat[mat_num] = [E11, E22, nu12, G12, X11t, X11c, X22t, X22c, X12]
Mat = {
    1: [181000, 10300, 0.28, 7170, 1500, 1500, 40, 246, 68],
}


#### Definition of the Laminate Stacking Sequence

- The LSS is defined through material (previously defined), orientation (in degrees) and thickness (in mm).
- The first layer is assumed to be the one on the bottom, according to the Z direction.
- Define as many layers as required following this structure: `L[layer_num] = [mat_num, orientation, thickness]`.
- If the laminate is symmetric, set `LSS_symm = True` and only define the first half of the LSS. For LSS with odd symmetry, define the central ply with half of the thickness.

In [ ]:
LSS_symm = False

# L[layer_num] = [mat_num, orientation (deg), thickness (mm)]
L = np.array([
    [1,  0, 2.0],
    [1, 45, 1.0],
])


#### Definition of the applied unit forces and moments

Define the applied unit forces (N/mm) and unit moments (N) following this structure: `NM = [Nxx, Nyy, Nxy, Mxx, Myy, Mxy]`.

In [ ]:
NM = np.array([50.0, 0.0, 0.0, 0.0, 0.0, 0.0])


## Calculations

First it is necessary to complete the LSS in case a symmetric LSS is defined:

In [ ]:
n_lam = L.shape[0]

if LSS_symm:
    L = np.vstack([L, L[::-1]])

n_lam = L.shape[0]
L


#### Calculation and assembly of stiffness, compliance and transformation matrices

The plane-stress compliance matrix S is calculated for each ply according to the definition for a transversely isotropic material. The plane-stress stiffness matrix Q is obtained by inverting S:

$$
S = \begin{bmatrix}
1/E_{11} & -\nu_{12}/E_{11} & 0 \\
-\nu_{12}/E_{11} & 1/E_{22} & 0 \\
0 & 0 & 1/G_{12}
\end{bmatrix}
$$

In [ ]:
S = np.zeros((3, 3, n_lam))
Q = np.zeros((3, 3, n_lam))

for i in range(n_lam):
    i_mat = int(L[i, 0])
    E11, E22, nu12, G12 = Mat[i_mat][:4]

    S[0, 0, i] = 1 / E11
    S[0, 1, i] = -nu12 / E11
    S[1, 0, i] = S[0, 1, i]
    S[1, 1, i] = 1 / E22
    S[2, 2, i] = 1 / G12

    Q[:, :, i] = np.linalg.inv(S[:, :, i])


The transformation matrix T is calculated for each ply according to the definition. The transformation matrix for strain, `Tg`, is also obtained according to the definition.

In [ ]:
T = np.zeros((3, 3, n_lam))
Tg = np.zeros((3, 3, n_lam))

for i in range(n_lam):
    theta = L[i, 1] * np.pi / 180
    m = np.cos(theta)
    n = np.sin(theta)

    T[0, 0, i] = m**2
    T[0, 1, i] = n**2
    T[0, 2, i] = 2 * m * n
    T[1, 0, i] = n**2
    T[1, 1, i] = m**2
    T[1, 2, i] = -2 * m * n
    T[2, 0, i] = -m * n
    T[2, 1, i] = m * n
    T[2, 2, i] = m**2 - n**2

    Tg[:, :, i] = np.linalg.inv(T[:, :, i]).T


The transformed compliance and stiffness matrices are obtained for each ply:

In [ ]:
Sb = np.zeros((3, 3, n_lam))
Qb = np.zeros((3, 3, n_lam))

for i in range(n_lam):
    Sb[:, :, i] = np.linalg.inv(Tg[:, :, i]) @ S[:, :, i] @ T[:, :, i]
    Qb[:, :, i] = np.linalg.inv(Sb[:, :, i])


### Calculation of the laminate's constitutive equation

Before determining matrices A, B and D, it is necessary to determine the total thickness of the laminate and the coordinates of each ply surface from the laminate's mid-plane:

In [ ]:
TH = L[:, 2].sum()
th = TH / 2

Z = np.zeros(n_lam + 1)
Z[0] = -th
running = 0.0
for i in range(n_lam):
    running += L[i, 2]
    Z[i + 1] = running - th

Z


#### Calculation of matrices A, B and D

Matrices A, B and D are calculated taking into account the bottom and top coordinates of the layers and the corresponding stiffness matrices:

$$
A_{jk} = \sum_{i=1}^{n} \bar{Q}_{jk,i} (Z_{i+1} - Z_i) \qquad
B_{jk} = \sum_{i=1}^{n} \bar{Q}_{jk,i} \frac{Z_{i+1}^2 - Z_i^2}{2} \qquad
D_{jk} = \sum_{i=1}^{n} \bar{Q}_{jk,i} \frac{Z_{i+1}^3 - Z_i^3}{3}
$$

In [ ]:
A = np.zeros((3, 3))
B = np.zeros((3, 3))
D = np.zeros((3, 3))

for i in range(n_lam):
    A += Qb[:, :, i] * (Z[i + 1] - Z[i])
    B += Qb[:, :, i] * (Z[i + 1]**2 - Z[i]**2) / 2
    D += Qb[:, :, i] * (Z[i + 1]**3 - Z[i]**3) / 3

ABD = np.block([[A, B], [B, D]])
ABD


#### Calculation of the laminate's strains and stresses

First it is necessary to determine the laminate's mid-plane strains and curvatures, inverting matrix ABD and multiplying by the unit forces and moments vector:

In [ ]:
eps_kappa = np.linalg.inv(ABD) @ NM
eps_kappa


Strains and stresses at the bottom and top locations of each ply are calculated taking into account the laminate's mid-plane strains and curvatures and the locations of each ply.

In [ ]:
eps_xyz = np.zeros((3, 2, n_lam))
eps_123 = np.zeros((3, 2, n_lam))
sigma_xyz = np.zeros((3, 2, n_lam))
sigma_123 = np.zeros((3, 2, n_lam))

for i in range(n_lam):
    eps_xyz[:, 0, i] = eps_kappa[0:3] + Z[i] * eps_kappa[3:6]
    eps_xyz[:, 1, i] = eps_kappa[0:3] + Z[i + 1] * eps_kappa[3:6]
    eps_123[:, :, i] = Tg[:, :, i] @ eps_xyz[:, :, i]
    sigma_123[:, :, i] = Q[:, :, i] @ eps_123[:, :, i]
    sigma_xyz[:, :, i] = np.linalg.inv(T[:, :, i]) @ sigma_123[:, :, i]


#### Calculation of the Failure Index according to the Tsai-Wu failure criterion

The failure index is calculated according to the 2D form of the Tsai-Wu criterion for each ply at bottom and top positions as:

$$
FI = \left(\frac{1}{X_{11t}} - \frac{1}{X_{11c}}\right)\sigma_1 + \left(\frac{1}{X_{22t}} - \frac{1}{X_{22c}}\right)\sigma_2
- \frac{\sigma_1 \sigma_2}{\sqrt{X_{11t}X_{11c}X_{22t}X_{22c}}}
+ \frac{\sigma_1^2}{X_{11t}X_{11c}} + \frac{\sigma_2^2}{X_{22t}X_{22c}} + \frac{\sigma_6^2}{X_{12}^2}
$$

In [ ]:
FI = np.zeros((2, n_lam))

for i in range(n_lam):
    i_mat = int(L[i, 0])
    _, _, _, _, X11t, X11c, X22t, X22c, X12 = Mat[i_mat]
    for j in range(2):
        s1 = sigma_123[0, j, i]
        s2 = sigma_123[1, j, i]
        s6 = sigma_123[2, j, i]

        A1 = s1**2 / (X11t * X11c)
        A2 = s2**2 / (X22t * X22c)
        A6 = s6**2 / (X12 * X12)
        A12 = -s1 * s2 / np.sqrt(X11t * X11c * X22t * X22c)
        BB1 = (1 / X11t - 1 / X11c) * s1
        BB2 = (1 / X22t - 1 / X22c) * s2

        FI[j, i] = BB1 + BB2 + A12 + A1 + A2 + A6


In addition to the failure index, the inverse reserve factor and the reserve factor can be also calculated:

$$
IRF = \sqrt{FI} \qquad RF = \frac{1}{IRF}
$$

Note: when `FI` is negative, its square root is a complex number, matching the behavior of the original MATLAB script (which also produces complex `IRF`/`RF` values, with the imaginary part ignored when plotting).

In [ ]:
IRF = np.emath.sqrt(FI)  # complex-aware sqrt, matches MATLAB's behaviour for negative FI
RF = 1 / IRF


## Results

#### Local and global strain

The resulting distribution of strain in local coordinates at the bottom and top positions of each ply along the laminate thickness is:

In [ ]:
#@title { display-mode: "form" }
eps_loc = np.zeros((2 * n_lam, 3))
eps_glob = np.zeros((2 * n_lam, 3))
ZZ = np.zeros(2 * n_lam)
layer = np.zeros(2 * n_lam, dtype=int)
pos = [''] * (2 * n_lam)

for i in range(n_lam):
    j = 2 * i
    idx_bot = 2 * n_lam - j - 1   # 0-based equivalent of MATLAB's (2*n_lam - j)
    idx_top = 2 * n_lam - j - 2   # 0-based equivalent of MATLAB's (2*n_lam - j - 1)

    ZZ[idx_bot] = Z[i]
    ZZ[idx_top] = Z[i + 1]
    layer[idx_bot] = i + 1
    layer[idx_top] = i + 1
    pos[idx_bot] = 'Bot'
    pos[idx_top] = 'Top'

    eps_loc[idx_bot, :] = eps_123[:, 0, i]
    eps_loc[idx_top, :] = eps_123[:, 1, i]
    eps_glob[idx_bot, :] = eps_xyz[:, 0, i]
    eps_glob[idx_top, :] = eps_xyz[:, 1, i]

Eps_loc = pd.DataFrame({
    'Layer': layer,
    'Position': pos,
    'e11 [-]': eps_loc[:, 0],
    'e22 [-]': eps_loc[:, 1],
    'e12 [-]': eps_loc[:, 2],
    'exx [-]': eps_glob[:, 0],
    'eyy [-]': eps_glob[:, 1],
    'exy [-]': eps_glob[:, 2],
})
Eps_loc


In [ ]:
#@title { display-mode: "form" }
fig, ax = plt.subplots()
for k, label in enumerate(['Strain11', 'Strain22', 'Strain12']):
    ax.plot(eps_loc[:, k], ZZ, marker='o', label=label)
ax.set_xlabel('Strain (m/m)')
ax.set_ylabel('Laminate thickness (mm)')
ax.legend()
ax.grid(True)
plt.show()


In [ ]:
#@title { display-mode: "form" }
fig, ax = plt.subplots()
for k, label in enumerate(['Strainxx', 'Strainyy', 'Strainxy']):
    ax.plot(eps_glob[:, k], ZZ, marker='o', label=label)
ax.set_xlabel('Strain (m/m)')
ax.set_ylabel('Laminate thickness (mm)')
ax.legend()
ax.grid(True)
plt.show()


#### Local and global stress

The resulting distribution of stress in local coordinates at the bottom and top positions of each ply along the laminate thickness is:

In [ ]:
#@title { display-mode: "form" }
sigma_loc = np.zeros((2 * n_lam, 3))
sigma_glob = np.zeros((2 * n_lam, 3))

for i in range(n_lam):
    j = 2 * i
    idx_bot = 2 * n_lam - j - 1
    idx_top = 2 * n_lam - j - 2

    sigma_loc[idx_bot, :] = sigma_123[:, 0, i]
    sigma_loc[idx_top, :] = sigma_123[:, 1, i]
    sigma_glob[idx_bot, :] = sigma_xyz[:, 0, i]
    sigma_glob[idx_top, :] = sigma_xyz[:, 1, i]

Sigma_loc = pd.DataFrame({
    'Layer': layer,
    'Position': pos,
    's11 [MPa]': sigma_loc[:, 0],
    's22 [MPa]': sigma_loc[:, 1],
    's12 [MPa]': sigma_loc[:, 2],
    'sxx [MPa]': sigma_glob[:, 0],
    'syy [MPa]': sigma_glob[:, 1],
    'sxy [MPa]': sigma_glob[:, 2],
})
Sigma_loc


In [ ]:
#@title { display-mode: "form" }
fig, ax = plt.subplots()
for k, label in enumerate(['Sigma11', 'Sigma22', 'Sigma12']):
    ax.plot(sigma_loc[:, k], ZZ, marker='o', label=label)
ax.set_xlabel('Stress (MPa)')
ax.set_ylabel('Laminate thickness (mm)')
ax.legend()
ax.grid(True)
plt.show()


In [ ]:
#@title { display-mode: "form" }
fig, ax = plt.subplots()
for k, label in enumerate(['Sigmaxx', 'Sigmayy', 'Sigmaxy']):
    ax.plot(sigma_glob[:, k], ZZ, marker='o', label=label)
ax.set_xlabel('Stress (MPa)')
ax.set_ylabel('Laminate thickness (mm)')
ax.legend()
ax.grid(True)
plt.show()


#### Failure index

The resulting distribution of failure index, inverse reserve factor and reserve factor at the bottom and top positions of each ply along the laminate thickness is:

In [ ]:
#@title { display-mode: "form" }
FI_plot = np.zeros(2 * n_lam)
IRF_plot = np.zeros(2 * n_lam, dtype=complex)
RF_plot = np.zeros(2 * n_lam, dtype=complex)

for i in range(n_lam):
    j = 2 * i
    idx_bot = 2 * n_lam - j - 1
    idx_top = 2 * n_lam - j - 2

    FI_plot[idx_bot] = FI[0, i]
    FI_plot[idx_top] = FI[1, i]

IRF_plot = np.emath.sqrt(FI_plot)
RF_plot = 1 / IRF_plot

FI_res = pd.DataFrame({
    'Layer': layer,
    'Position': pos,
    'FI': FI_plot,
    'IRF': IRF_plot,
    'RF': RF_plot,
})
FI_res


In [ ]:
#@title { display-mode: "form" }
fig, ax = plt.subplots()
ax.plot(FI_plot, ZZ, marker='o', label='Failure Index')
ax.plot(IRF_plot.real, ZZ, marker='o', label='Inverse Reserve Factor')
ax.set_xlabel('Tsai-Wu Failure Index & Inverse Reserve Factor')
ax.set_ylabel('Laminate thickness (mm)')
ax.legend()
ax.grid(True)
plt.show()


In [ ]:
#@title { display-mode: "form" }
fig, ax = plt.subplots()
ax.plot(RF_plot.real, ZZ, marker='o')
ax.set_xlabel('Tsai-Wu Reserve Factor')
ax.set_ylabel('Laminate thickness (mm)')
ax.grid(True)
plt.show()
